In [1]:
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\train_raw.parquet으로부터
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_diff.parquet 생성

import duckdb
import os

# ─────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────
BASE_DIR   = r"C:\Workspace\06_ML_projdect\26_1_COIN"
SPLIT_DIR  = os.path.join(BASE_DIR, "data2", "03_splitting")
OUT_DIR    = os.path.join(BASE_DIR, "data2", "04_feature_engineering")
# === 수정: 오직 훈련(train_raw) 데이터만 타겟으로 지정 ===
IN_FILE    = os.path.join(SPLIT_DIR, "train_raw.parquet")
OUT_FILE   = os.path.join(OUT_DIR, "fs_sample_diff.parquet")

os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# 원본 컬럼 → 차분 별칭 매핑 (18개)
# 규칙: smart_X_raw → sX_diff / 나머지 → 소문자_diff
# ─────────────────────────────────────────────
DIFF_COL_ALIAS = {
    "smart_5_raw":       "s5_diff",
    "smart_184_raw":     "s184_diff",
    "smart_187_raw":     "s187_diff",
    "smart_197_raw":     "s197_diff",
    "smart_198_raw":     "s198_diff",
    "timeout_5s":        "timeout_5s_diff",
    "timeout_total":     "timeout_total_diff",
    "seek_error_count":  "seek_error_count_diff",
    "smart_189_raw":     "s189_diff",
    "smart_191_raw":     "s191_diff",
    "smart_194_raw":     "s194_diff",
    "smart_199_raw":     "s199_diff",
    "smart_241_raw":     "s241_diff",
    "smart_242_raw":     "s242_diff",
    "total_reads":       "total_reads_diff",
    "total_seeks":       "total_seeks_diff",
    "smart_183_raw":     "s183_diff",
    "smart_190_raw":     "s190_diff",
}

# ─────────────────────────────────────────────
# 원본 19개 컬럼
# ─────────────────────────────────────────────
RAW_COLS = [
    "smart_5_raw",
    "smart_184_raw",
    "smart_187_raw",
    "smart_197_raw",
    "smart_198_raw",
    "timeout_5s",
    "timeout_total",
    "seek_error_count",
    "smart_9_raw",
    "smart_189_raw",
    "smart_191_raw",
    "smart_194_raw",
    "smart_199_raw",
    "smart_241_raw",
    "smart_242_raw",
    "total_reads",
    "total_seeks",
    "smart_183_raw",
    "smart_190_raw",
]

# ─────────────────────────────────────────────
# SQL 구성
# ─────────────────────────────────────────────
in_path  = IN_FILE.replace("\\", "/")
out_path = OUT_FILE.replace("\\", "/")

raw_select  = ",\n        ".join(f"r.{c}" for c in RAW_COLS)

# 가짜 스파이크 방지: 첫 행은 순수하게 NULL 상태로 둡니다.
diff_select = ",\n        ".join(
    f"r.{col} - LAG(r.{col}) OVER w AS {alias}"
    for col, alias in DIFF_COL_ALIAS.items()
)

sql = f"""
COPY (
    WITH raw AS (
        SELECT
            serial_number,
            date,
            failure,
            {raw_select},
            {diff_select}
        FROM read_parquet('{in_path}') AS r
        WINDOW w AS (
            PARTITION BY serial_number
            ORDER BY date
        )
    )
    SELECT *
    FROM raw
    ORDER BY serial_number, date
) TO '{out_path}' (FORMAT PARQUET, COMPRESSION 'zstd');
"""

print("=== 실행할 SQL ===")
print(sql)

# ─────────────────────────────────────────────
# 실행
# ─────────────────────────────────────────────
con = duckdb.connect()
con.execute(sql)
con.close()
print(f"✅ 저장 완료: {OUT_FILE}")

# ─────────────────────────────────────────────
# 검증
# ─────────────────────────────────────────────
con = duckdb.connect()

info = con.execute(f"""
    SELECT
        COUNT(*)                                      AS total_rows,
        COUNT(DISTINCT serial_number)                 AS unique_serials,
        SUM(CASE WHEN failure = 1 THEN 1 ELSE 0 END)  AS failure_rows
    FROM read_parquet('{out_path}')
""").fetchdf()

cols = con.execute(f"""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('{out_path}'))
""").fetchdf()

con.close()

print("=== 기본 통계 ===")
print(info.to_string(index=False))
print(f"\n=== 컬럼 목록 (총 {len(cols)}개) ===")
print(cols.to_string(index=False))

=== 실행할 SQL ===

COPY (
    WITH raw AS (
        SELECT
            serial_number,
            date,
            failure,
            r.smart_5_raw,
        r.smart_184_raw,
        r.smart_187_raw,
        r.smart_197_raw,
        r.smart_198_raw,
        r.timeout_5s,
        r.timeout_total,
        r.seek_error_count,
        r.smart_9_raw,
        r.smart_189_raw,
        r.smart_191_raw,
        r.smart_194_raw,
        r.smart_199_raw,
        r.smart_241_raw,
        r.smart_242_raw,
        r.total_reads,
        r.total_seeks,
        r.smart_183_raw,
        r.smart_190_raw,
            r.smart_5_raw - LAG(r.smart_5_raw) OVER w AS s5_diff,
        r.smart_184_raw - LAG(r.smart_184_raw) OVER w AS s184_diff,
        r.smart_187_raw - LAG(r.smart_187_raw) OVER w AS s187_diff,
        r.smart_197_raw - LAG(r.smart_197_raw) OVER w AS s197_diff,
        r.smart_198_raw - LAG(r.smart_198_raw) OVER w AS s198_diff,
        r.timeout_5s - LAG(r.timeout_5s) OVER w AS timeout_5s_diff,


# fs_sample_7d.parquet 생성

In [2]:
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\train_raw.parquet와
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_diff.parquet으로부터
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_7d.parquet 생성

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import math

# 1. 파일 경로 설정
BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN"
IN_FILE  = os.path.join(BASE_DIR, "data2", "04_feature_engineering", "fs_sample_diff.parquet")
OUT_FILE = os.path.join(BASE_DIR, "data2", "04_feature_engineering", "fs_sample_7d.parquet")

if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

# 2. 파케이 자체 메타데이터(Zone-Map)를 활용한 초고속 시리얼 스캔 (디스크 복사 X)
print("1/3. 원본 파케이(Parquet)에서 고유 시리얼 스캔 중...")
con = duckdb.connect()  # 임시 파일 없이 메모리에서만 가볍게 실행
serials = con.execute(f"SELECT DISTINCT serial_number FROM read_parquet('{IN_FILE.replace(chr(92), "/")}')").df()['serial_number'].tolist()

# 3. Chunk 단위 처리
CHUNK_SIZE = 1000
total_chunks = math.ceil(len(serials) / CHUNK_SIZE)
print(f"2/4. 총 {len(serials):,}개의 디스크를 {total_chunks}개의 Chunk로 안전하게 처리합니다.")

writer = None

for chunk_idx, i in enumerate(range(0, len(serials), CHUNK_SIZE)):
    chunk_serials = serials[i : i + CHUNK_SIZE]
    
    # 타입 방어적 문자열 변환
    serial_str = ", ".join([f"'{s}'" if isinstance(s, str) else str(s) for s in chunk_serials])

    # 4. DuckDB 조각 SQL (20개 에러 피처 및 기존 피처 통합)
    sql = f"""
    WITH base AS (
        SELECT 
            serial_number, date,
            -- [추가] 에러 집계 대상 (diff)
            s5_diff, s187_diff, s197_diff, s198_diff,
            s183_diff, s191_diff, s199_diff,
            timeout_5s_diff, timeout_total_diff, seek_error_count_diff,
            s184_diff, 
            -- 기존 대상
            smart_190_raw, smart_194_raw, 
            s241_diff, s242_diff, 
            total_reads_diff, total_seeks_diff
        FROM read_parquet('{IN_FILE.replace(chr(92), "/")}') 
        WHERE serial_number IN (SELECT unnest([{serial_str}]))
    ),
    diffs AS (
        SELECT *,
            smart_190_raw - LAG(smart_190_raw) OVER w0 AS d_s190,
            smart_194_raw - LAG(smart_194_raw) OVER w0 AS d_s194,
            s241_diff - LAG(s241_diff) OVER w0 AS d_s241,
            s242_diff - LAG(s242_diff) OVER w0 AS d_s242,
            total_reads_diff - LAG(total_reads_diff) OVER w0 AS d_reads,
            total_seeks_diff - LAG(total_seeks_diff) OVER w0 AS d_seeks
        FROM base
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    accel_base AS (
        SELECT *,
            d_s241 - LAG(d_s241) OVER w0 AS s241_7d_accel,
            d_s242 - LAG(d_s242) OVER w0 AS s242_7d_accel,
            d_reads - LAG(d_reads) OVER w0 AS total_reads_7d_accel,
            d_seeks - LAG(d_seeks) OVER w0 AS total_seeks_7d_accel
        FROM diffs
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    stats_7d AS (
        SELECT
            serial_number, date,

            -- 3일/7일 기존 통계
            MAX(s184_diff) OVER w3 AS s184_3d_max, SUM(s184_diff) OVER w3 AS s184_3d_sum,
            MAX(s184_diff) OVER w7 AS s184_7d_max, SUM(s184_diff) OVER w7 AS s184_7d_sum,

            MAX(smart_190_raw) OVER w7 AS s190_7d_max, AVG(smart_190_raw) OVER w7 AS s190_7d_mean,
            COALESCE(STDDEV_SAMP(smart_190_raw) OVER w7, 0.0) AS s190_7d_std,
            SUM(ABS(d_s190)) OVER w7 AS s190_7d_asfd, SQRT(SUM(POW(d_s190, 2)) OVER w7) AS s190_7d_cid,
            (smart_190_raw - FIRST_VALUE(smart_190_raw) OVER w7_dai) / NULLIF(COUNT(smart_190_raw) OVER w7_dai - 1, 0) AS s190_7d_dai,

            MAX(smart_194_raw) OVER w7 AS s194_7d_max, AVG(smart_194_raw) OVER w7 AS s194_7d_mean,
            COALESCE(STDDEV_SAMP(smart_194_raw) OVER w7, 0.0) AS s194_7d_std,
            SUM(ABS(d_s194)) OVER w7 AS s194_7d_asfd, SQRT(SUM(POW(d_s194, 2)) OVER w7) AS s194_7d_cid,
            (smart_194_raw - FIRST_VALUE(smart_194_raw) OVER w7_dai) / NULLIF(COUNT(smart_194_raw) OVER w7_dai - 1, 0) AS s194_7d_dai,

            MAX(s241_diff) OVER w7 AS s241_7d_max, SUM(s241_diff) OVER w7 AS s241_7d_sum, AVG(s241_diff) OVER w7 AS s241_7d_mean,
            COALESCE(STDDEV_SAMP(s241_diff) OVER w7, 0.0) AS s241_7d_std, SUM(ABS(d_s241)) OVER w7 AS s241_7d_asfd,
            (s241_diff - FIRST_VALUE(s241_diff) OVER w7_dai) / NULLIF(COUNT(s241_diff) OVER w7_dai - 1, 0) AS s241_7d_dai,

            MAX(s242_diff) OVER w7 AS s242_7d_max, SUM(s242_diff) OVER w7 AS s242_7d_sum, AVG(s242_diff) OVER w7 AS s242_7d_mean,
            COALESCE(STDDEV_SAMP(s242_diff) OVER w7, 0.0) AS s242_7d_std, SUM(ABS(d_s242)) OVER w7 AS s242_7d_asfd,
            (s242_diff - FIRST_VALUE(s242_diff) OVER w7_dai) / NULLIF(COUNT(s242_diff) OVER w7_dai - 1, 0) AS s242_7d_dai,

            MAX(total_reads_diff) OVER w7 AS total_reads_7d_max, SUM(total_reads_diff) OVER w7 AS total_reads_7d_sum, AVG(total_reads_diff) OVER w7 AS total_reads_7d_mean,
            COALESCE(STDDEV_SAMP(total_reads_diff) OVER w7, 0.0) AS total_reads_7d_std, SUM(ABS(d_reads)) OVER w7 AS total_reads_7d_asfd,
            (total_reads_diff - FIRST_VALUE(total_reads_diff) OVER w7_dai) / NULLIF(COUNT(total_reads_diff) OVER w7_dai - 1, 0) AS total_reads_7d_dai,

            MAX(total_seeks_diff) OVER w7 AS total_seeks_7d_max, SUM(total_seeks_diff) OVER w7 AS total_seeks_7d_sum, AVG(total_seeks_diff) OVER w7 AS total_seeks_7d_mean,
            COALESCE(STDDEV_SAMP(total_seeks_diff) OVER w7, 0.0) AS total_seeks_7d_std, SUM(ABS(d_seeks)) OVER w7 AS total_seeks_7d_asfd,
            (total_seeks_diff - FIRST_VALUE(total_seeks_diff) OVER w7_dai) / NULLIF(COUNT(total_seeks_diff) OVER w7_dai - 1, 0) AS total_seeks_7d_dai,

            smart_190_raw AS r_190, smart_194_raw AS r_194, 
            s241_diff AS r_241, s242_diff AS r_242, 
            total_reads_diff AS r_reads, total_seeks_diff AS r_seeks,
            s241_7d_accel, s242_7d_accel, total_reads_7d_accel, total_seeks_7d_accel
        FROM accel_base
        WINDOW 
            w3 AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),
            w7 AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),
            w7_dai AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
    )
    SELECT
        serial_number, date,
        -- 에러 집계 출력
        
        s184_3d_max, s184_3d_sum, s184_7d_max, s184_7d_sum,
        s190_7d_max, s190_7d_mean, s190_7d_std, s190_7d_asfd, s190_7d_cid, s190_7d_dai,
        CASE WHEN s190_7d_std < 0.001 THEN 0.0 ELSE (r_190 - s190_7d_mean) / (s190_7d_std + 1e-5) END AS s190_7d_zscore,
        s194_7d_max, s194_7d_mean, s194_7d_std, s194_7d_asfd, s194_7d_cid, s194_7d_dai,
        CASE WHEN s194_7d_std < 0.001 THEN 0.0 ELSE (r_194 - s194_7d_mean) / (s194_7d_std + 1e-5) END AS s194_7d_zscore,
        s241_7d_max, s241_7d_sum, s241_7d_mean, s241_7d_std, s241_7d_asfd, s241_7d_dai, s241_7d_accel,
        CASE WHEN s241_7d_std < 0.001 THEN 0.0 ELSE (r_241 - s241_7d_mean) / (s241_7d_std + 1e-5) END AS s241_7d_zscore,
        s242_7d_max, s242_7d_sum, s242_7d_mean, s242_7d_std, s242_7d_asfd, s242_7d_dai, s242_7d_accel,
        CASE WHEN s242_7d_std < 0.001 THEN 0.0 ELSE (r_242 - s242_7d_mean) / (s242_7d_std + 1e-5) END AS s242_7d_zscore,
        total_reads_7d_max, total_reads_7d_sum, total_reads_7d_mean, total_reads_7d_std, total_reads_7d_asfd, total_reads_7d_dai, total_reads_7d_accel,
        CASE WHEN total_reads_7d_std < 0.001 THEN 0.0 ELSE (r_reads - total_reads_7d_mean) / (total_reads_7d_std + 1e-5) END AS total_reads_7d_zscore,
        total_seeks_7d_max, total_seeks_7d_sum, total_seeks_7d_mean, total_seeks_7d_std, total_seeks_7d_asfd, total_seeks_7d_dai, total_seeks_7d_accel,
        CASE WHEN total_seeks_7d_std < 0.001 THEN 0.0 ELSE (r_seeks - total_seeks_7d_mean) / (total_seeks_7d_std + 1e-5) END AS total_seeks_7d_zscore,
        r_190, r_194, r_241, r_242, r_reads, r_seeks
    FROM stats_7d
    """

    
    df = con.query(sql).df()

    df = df.fillna(0.0)

    # EWMA 연산 오류 방지용 엄격한 시계열 정렬 적용
    df.sort_values(by=['serial_number', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # EWMA 계산 (span=7)
    ewma_targets = {
        'r_190': 's190_7d_ewma', 'r_194': 's194_7d_ewma', 'r_241': 's241_7d_ewma',
        'r_242': 's242_7d_ewma', 'r_reads': 'total_reads_7d_ewma', 'r_seeks': 'total_seeks_7d_ewma'
    }
    for col_raw, col_ewma in ewma_targets.items():
        df[col_ewma] = df.groupby('serial_number')[col_raw].transform(lambda x: x.ewm(span=7, adjust=False).mean())
    df.drop(columns=list(ewma_targets.keys()), inplace=True)

    # 강제 메모리 압축 (Float64 -> Float32) 단, 정밀도 필수 컬럼은 제외!
    exclude_keywords = ['241', '242', 'reads', 'seeks', 'cid']
    float_cols = df.select_dtypes(include=['float64']).columns
    target_cols = [c for c in float_cols if not any(k in c.lower() for k in exclude_keywords)]
    
    df[target_cols] = df[target_cols].astype('float32')

    # Parquet 디스크 쓰기
    table = pa.Table.from_pandas(df)
    if writer is None:
        writer = pq.ParquetWriter(OUT_FILE, table.schema, compression='zstd')
    writer.write_table(table)
    
    del df, table
    gc.collect()
    
    print(f"진행 상황: [{chunk_idx + 1}/{total_chunks}] 7d 조각 완료")

if writer:
    writer.close()
con.close()

print(f"3/3. 🎉 디스크 100% 이슈가 완벽히 해결된 7일 파이프라인 종료! (저장: {OUT_FILE})")

1/3. 원본 파케이(Parquet)에서 고유 시리얼 스캔 중...
2/4. 총 53,057개의 디스크를 54개의 Chunk로 안전하게 처리합니다.
진행 상황: [1/54] 7d 조각 완료
진행 상황: [2/54] 7d 조각 완료
진행 상황: [3/54] 7d 조각 완료
진행 상황: [4/54] 7d 조각 완료
진행 상황: [5/54] 7d 조각 완료
진행 상황: [6/54] 7d 조각 완료
진행 상황: [7/54] 7d 조각 완료
진행 상황: [8/54] 7d 조각 완료
진행 상황: [9/54] 7d 조각 완료
진행 상황: [10/54] 7d 조각 완료
진행 상황: [11/54] 7d 조각 완료
진행 상황: [12/54] 7d 조각 완료
진행 상황: [13/54] 7d 조각 완료
진행 상황: [14/54] 7d 조각 완료
진행 상황: [15/54] 7d 조각 완료
진행 상황: [16/54] 7d 조각 완료
진행 상황: [17/54] 7d 조각 완료
진행 상황: [18/54] 7d 조각 완료
진행 상황: [19/54] 7d 조각 완료
진행 상황: [20/54] 7d 조각 완료
진행 상황: [21/54] 7d 조각 완료
진행 상황: [22/54] 7d 조각 완료
진행 상황: [23/54] 7d 조각 완료
진행 상황: [24/54] 7d 조각 완료
진행 상황: [25/54] 7d 조각 완료
진행 상황: [26/54] 7d 조각 완료
진행 상황: [27/54] 7d 조각 완료
진행 상황: [28/54] 7d 조각 완료
진행 상황: [29/54] 7d 조각 완료
진행 상황: [30/54] 7d 조각 완료
진행 상황: [31/54] 7d 조각 완료
진행 상황: [32/54] 7d 조각 완료
진행 상황: [33/54] 7d 조각 완료
진행 상황: [34/54] 7d 조각 완료
진행 상황: [35/54] 7d 조각 완료
진행 상황: [36/54] 7d 조각 완료
진행 상황: [37/54] 7d 조각 완료
진행 상황: [38/54] 7d 조각 완료
진행 상황: [39/54]

# fs_sample_14d.parquet 생성

In [3]:
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\train_raw.parquet와
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_diff.parquet으로부터
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_14d.parquet 생성

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import math

# 1. 파일 경로 설정
BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN"
IN_FILE  = os.path.join(BASE_DIR, "data2", "04_feature_engineering", "fs_sample_diff.parquet")
OUT_FILE = os.path.join(BASE_DIR, "data2", "04_feature_engineering", "fs_sample_14d.parquet")

if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

# 2. 파케이 자체 메타데이터(Zone-Map)를 활용한 초고속 시리얼 스캔 (디스크 복사 X)
print("1/3. 원본 파케이(Parquet)에서 고유 시리얼 스캔 중...")
con = duckdb.connect()  # 임시 파일 없이 메모리에서만 가볍게 실행
serials = con.execute(f"SELECT DISTINCT serial_number FROM read_parquet('{IN_FILE.replace(chr(92), "/")}')").df()['serial_number'].tolist()

# 3. Chunk 단위 처리
CHUNK_SIZE = 1000
total_chunks = math.ceil(len(serials) / CHUNK_SIZE)
print(f"2/4. 총 {len(serials):,}개의 디스크를 {total_chunks}개의 Chunk로 안전하게 처리합니다.")

writer = None

for chunk_idx, i in enumerate(range(0, len(serials), CHUNK_SIZE)):
    chunk_serials = serials[i : i + CHUNK_SIZE]
    
    # 타입 방어적 문자열 변환
    serial_str = ", ".join([f"'{s}'" if isinstance(s, str) else str(s) for s in chunk_serials])
    
        # 4. DuckDB 조각 SQL (참조 오류까지 모두 제거 완료)
    sql = f"""
    WITH base AS (
        SELECT 
            serial_number, date,
            s5_diff, s187_diff, s197_diff, s198_diff,
            s183_diff, s191_diff, s199_diff,
            timeout_5s_diff, timeout_total_diff, seek_error_count_diff,
            s184_diff, 
            smart_190_raw, smart_194_raw, 
            s241_diff, s242_diff, 
            total_reads_diff, total_seeks_diff
        FROM read_parquet('{IN_FILE.replace(chr(92), "/")}') 
        WHERE serial_number IN (SELECT unnest([{serial_str}]))
    ),
    diffs AS (
        SELECT *,
            smart_190_raw - LAG(smart_190_raw) OVER w0 AS d_s190,
            smart_194_raw - LAG(smart_194_raw) OVER w0 AS d_s194,
            s241_diff - LAG(s241_diff) OVER w0 AS d_s241,
            s242_diff - LAG(s242_diff) OVER w0 AS d_s242,
            total_reads_diff - LAG(total_reads_diff) OVER w0 AS d_reads,
            total_seeks_diff - LAG(total_seeks_diff) OVER w0 AS d_seeks
        FROM base
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    accel_base AS (
        SELECT *,
            d_s241 - LAG(d_s241) OVER w0 AS s241_14d_accel,
            d_s242 - LAG(d_s242) OVER w0 AS s242_14d_accel,
            d_reads - LAG(d_reads) OVER w0 AS total_reads_14d_accel,
            d_seeks - LAG(d_seeks) OVER w0 AS total_seeks_14d_accel
        FROM diffs
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    stats_14d AS (
        SELECT
            serial_number, date,
            SUM(s5_diff) OVER w14 AS s5_14d_sum, MAX(s5_diff) OVER w14 AS s5_14d_max,
            SUM(s187_diff) OVER w14 AS s187_14d_sum, MAX(s187_diff) OVER w14 AS s187_14d_max,
            SUM(s197_diff) OVER w14 AS s197_14d_sum, MAX(s197_diff) OVER w14 AS s197_14d_max,
            SUM(s198_diff) OVER w14 AS s198_14d_sum, MAX(s198_diff) OVER w14 AS s198_14d_max,
            SUM(s183_diff) OVER w14 AS s183_14d_sum, MAX(s183_diff) OVER w14 AS s183_14d_max,
            SUM(s191_diff) OVER w14 AS s191_14d_sum, MAX(s191_diff) OVER w14 AS s191_14d_max,
            SUM(s199_diff) OVER w14 AS s199_14d_sum, MAX(s199_diff) OVER w14 AS s199_14d_max,
            SUM(timeout_5s_diff) OVER w14 AS timeout_5s_14d_sum, MAX(timeout_5s_diff) OVER w14 AS timeout_5s_14d_max,
            SUM(timeout_total_diff) OVER w14 AS timeout_total_14d_sum, MAX(timeout_total_diff) OVER w14 AS timeout_total_14d_max,
            SUM(seek_error_count_diff) OVER w14 AS seek_error_count_14d_sum, MAX(seek_error_count_diff) OVER w14 AS seek_error_count_14d_max,
            MAX(s184_diff) OVER w14 AS s184_14d_max, SUM(s184_diff) OVER w14 AS s184_14d_sum,

            MAX(smart_190_raw) OVER w14 AS s190_14d_max, AVG(smart_190_raw) OVER w14 AS s190_14d_mean,
            COALESCE(STDDEV_SAMP(smart_190_raw) OVER w14, 0.0) AS s190_14d_std,
            SUM(ABS(d_s190)) OVER w14 AS s190_14d_asfd, SQRT(SUM(POW(d_s190, 2)) OVER w14) AS s190_14d_cid,
            (smart_190_raw - FIRST_VALUE(smart_190_raw) OVER w14_dai) / NULLIF(COUNT(smart_190_raw) OVER w14_dai - 1, 0) AS s190_14d_dai,

            MAX(smart_194_raw) OVER w14 AS s194_14d_max, AVG(smart_194_raw) OVER w14 AS s194_14d_mean,
            COALESCE(STDDEV_SAMP(smart_194_raw) OVER w14, 0.0) AS s194_14d_std,
            SUM(ABS(d_s194)) OVER w14 AS s194_14d_asfd, SQRT(SUM(POW(d_s194, 2)) OVER w14) AS s194_14d_cid,
            (smart_194_raw - FIRST_VALUE(smart_194_raw) OVER w14_dai) / NULLIF(COUNT(smart_194_raw) OVER w14_dai - 1, 0) AS s194_14d_dai,

            MAX(s241_diff) OVER w14 AS s241_14d_max, SUM(s241_diff) OVER w14 AS s241_14d_sum, AVG(s241_diff) OVER w14 AS s241_14d_mean,
            COALESCE(STDDEV_SAMP(s241_diff) OVER w14, 0.0) AS s241_14d_std, SUM(ABS(d_s241)) OVER w14 AS s241_14d_asfd,
            (s241_diff - FIRST_VALUE(s241_diff) OVER w14_dai) / NULLIF(COUNT(s241_diff) OVER w14_dai - 1, 0) AS s241_14d_dai,

            MAX(s242_diff) OVER w14 AS s242_14d_max, SUM(s242_diff) OVER w14 AS s242_14d_sum, AVG(s242_diff) OVER w14 AS s242_14d_mean,
            COALESCE(STDDEV_SAMP(s242_diff) OVER w14, 0.0) AS s242_14d_std, SUM(ABS(d_s242)) OVER w14 AS s242_14d_asfd,
            (s242_diff - FIRST_VALUE(s242_diff) OVER w14_dai) / NULLIF(COUNT(s242_diff) OVER w14_dai - 1, 0) AS s242_14d_dai,

            MAX(total_reads_diff) OVER w14 AS total_reads_14d_max, SUM(total_reads_diff) OVER w14 AS total_reads_14d_sum, AVG(total_reads_diff) OVER w14 AS total_reads_14d_mean,
            COALESCE(STDDEV_SAMP(total_reads_diff) OVER w14, 0.0) AS total_reads_14d_std, SUM(ABS(d_reads)) OVER w14 AS total_reads_14d_asfd,
            (total_reads_diff - FIRST_VALUE(total_reads_diff) OVER w14_dai) / NULLIF(COUNT(total_reads_diff) OVER w14_dai - 1, 0) AS total_reads_14d_dai,

            MAX(total_seeks_diff) OVER w14 AS total_seeks_14d_max, SUM(total_seeks_diff) OVER w14 AS total_seeks_14d_sum, AVG(total_seeks_diff) OVER w14 AS total_seeks_14d_mean,
            COALESCE(STDDEV_SAMP(total_seeks_diff) OVER w14, 0.0) AS total_seeks_14d_std, SUM(ABS(d_seeks)) OVER w14 AS total_seeks_14d_asfd,
            (total_seeks_diff - FIRST_VALUE(total_seeks_diff) OVER w14_dai) / NULLIF(COUNT(total_seeks_diff) OVER w14_dai - 1, 0) AS total_seeks_14d_dai,

            smart_190_raw AS r_190, smart_194_raw AS r_194, 
            s241_diff AS r_241, s242_diff AS r_242, 
            total_reads_diff AS r_reads, total_seeks_diff AS r_seeks,
            -- 가속도 전달 (온도 가속도 제외됨)
            s241_14d_accel, s242_14d_accel, total_reads_14d_accel, total_seeks_14d_accel
        FROM accel_base
        WINDOW 
            w0 AS (PARTITION BY serial_number ORDER BY date),
            w14 AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
            w14_dai AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW)
    )
    SELECT
        serial_number, date,
        s5_14d_sum, s5_14d_max, s187_14d_sum, s187_14d_max, s197_14d_sum, s197_14d_max,
        s198_14d_sum, s198_14d_max, s183_14d_sum, s183_14d_max, s191_14d_sum, s191_14d_max,
        s199_14d_sum, s199_14d_max, timeout_5s_14d_sum, timeout_5s_14d_max,
        timeout_total_14d_sum, timeout_total_14d_max, seek_error_count_14d_sum, seek_error_count_14d_max,
        s184_14d_max, s184_14d_sum,
        s190_14d_max, s190_14d_mean, s190_14d_std, s190_14d_asfd, s190_14d_cid, s190_14d_dai,
        CASE WHEN s190_14d_std < 0.001 THEN 0.0 ELSE (r_190 - s190_14d_mean) / (s190_14d_std + 1e-5) END AS s190_14d_zscore,
        s194_14d_max, s194_14d_mean, s194_14d_std, s194_14d_asfd, s194_14d_cid, s194_14d_dai,
        CASE WHEN s194_14d_std < 0.001 THEN 0.0 ELSE (r_194 - s194_14d_mean) / (s194_14d_std + 1e-5) END AS s194_14d_zscore,
        s241_14d_max, s241_14d_sum, s241_14d_mean, s241_14d_std, s241_14d_asfd, s241_14d_dai, s241_14d_accel,
        CASE WHEN s241_14d_std < 0.001 THEN 0.0 ELSE (r_241 - s241_14d_mean) / (s241_14d_std + 1e-5) END AS s241_14d_zscore,
        s242_14d_max, s242_14d_sum, s242_14d_mean, s242_14d_std, s242_14d_asfd, s242_14d_dai, s242_14d_accel,
        CASE WHEN s242_14d_std < 0.001 THEN 0.0 ELSE (r_242 - s242_14d_mean) / (s242_14d_std + 1e-5) END AS s242_14d_zscore,
        total_reads_14d_max, total_reads_14d_sum, total_reads_14d_mean, total_reads_14d_std, total_reads_14d_asfd, total_reads_14d_dai, total_reads_14d_accel,
        CASE WHEN total_reads_14d_std < 0.001 THEN 0.0 ELSE (r_reads - total_reads_14d_mean) / (total_reads_14d_std + 1e-5) END AS total_reads_14d_zscore,
        total_seeks_14d_max, total_seeks_14d_sum, total_seeks_14d_mean, total_seeks_14d_std, total_seeks_14d_asfd, total_seeks_14d_dai, total_seeks_14d_accel,
        CASE WHEN total_seeks_14d_std < 0.001 THEN 0.0 ELSE (r_seeks - total_seeks_14d_mean) / (total_seeks_14d_std + 1e-5) END AS total_seeks_14d_zscore,
        r_190, r_194, r_241, r_242, r_reads, r_seeks
    FROM stats_14d
    """



    
    df = con.query(sql).df()

    df = df.fillna(0.0)

    # EWMA 연산 오류 방지용 엄격한 시계열 정렬 적용
    df.sort_values(by=['serial_number', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # EWMA 계산 (span=14)
    ewma_targets = {
        'r_190': 's190_14d_ewma', 'r_194': 's194_14d_ewma', 'r_241': 's241_14d_ewma',
        'r_242': 's242_14d_ewma', 'r_reads': 'total_reads_14d_ewma', 'r_seeks': 'total_seeks_14d_ewma'
    }
    for col_raw, col_ewma in ewma_targets.items():
        df[col_ewma] = df.groupby('serial_number')[col_raw].transform(lambda x: x.ewm(span=14, adjust=False).mean())
    df.drop(columns=list(ewma_targets.keys()), inplace=True)

    # 강제 메모리 압축 (Float64 -> Float32) 단, 정밀도 필수 컬럼은 제외!
    exclude_keywords = ['241', '242', 'reads', 'seeks', 'cid']
    float_cols = df.select_dtypes(include=['float64']).columns
    target_cols = [c for c in float_cols if not any(k in c.lower() for k in exclude_keywords)]
    
    df[target_cols] = df[target_cols].astype('float32')

    # Parquet 디스크 쓰기
    table = pa.Table.from_pandas(df)
    if writer is None:
        writer = pq.ParquetWriter(OUT_FILE, table.schema, compression='zstd')
    writer.write_table(table)
    
    del df, table
    gc.collect()
    
    print(f"진행 상황: [{chunk_idx + 1}/{total_chunks}] 14d 조각 완료")

if writer:
    writer.close()
con.close()

print(f"3/3. 🎉 디스크 100% 이슈가 완벽히 해결된 14일 파이프라인 종료! (저장: {OUT_FILE})")

1/3. 원본 파케이(Parquet)에서 고유 시리얼 스캔 중...
2/4. 총 53,057개의 디스크를 54개의 Chunk로 안전하게 처리합니다.
진행 상황: [1/54] 14d 조각 완료
진행 상황: [2/54] 14d 조각 완료
진행 상황: [3/54] 14d 조각 완료
진행 상황: [4/54] 14d 조각 완료
진행 상황: [5/54] 14d 조각 완료
진행 상황: [6/54] 14d 조각 완료
진행 상황: [7/54] 14d 조각 완료
진행 상황: [8/54] 14d 조각 완료
진행 상황: [9/54] 14d 조각 완료
진행 상황: [10/54] 14d 조각 완료
진행 상황: [11/54] 14d 조각 완료
진행 상황: [12/54] 14d 조각 완료
진행 상황: [13/54] 14d 조각 완료
진행 상황: [14/54] 14d 조각 완료
진행 상황: [15/54] 14d 조각 완료
진행 상황: [16/54] 14d 조각 완료
진행 상황: [17/54] 14d 조각 완료
진행 상황: [18/54] 14d 조각 완료
진행 상황: [19/54] 14d 조각 완료
진행 상황: [20/54] 14d 조각 완료
진행 상황: [21/54] 14d 조각 완료
진행 상황: [22/54] 14d 조각 완료
진행 상황: [23/54] 14d 조각 완료
진행 상황: [24/54] 14d 조각 완료
진행 상황: [25/54] 14d 조각 완료
진행 상황: [26/54] 14d 조각 완료
진행 상황: [27/54] 14d 조각 완료
진행 상황: [28/54] 14d 조각 완료
진행 상황: [29/54] 14d 조각 완료
진행 상황: [30/54] 14d 조각 완료
진행 상황: [31/54] 14d 조각 완료
진행 상황: [32/54] 14d 조각 완료
진행 상황: [33/54] 14d 조각 완료
진행 상황: [34/54] 14d 조각 완료
진행 상황: [35/54] 14d 조각 완료
진행 상황: [36/54] 14d 조각 완료
진행 상황: [37/54] 14d 조각 완료
진

# fs_sample_28d.parquet 생성

In [ ]:
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\train_raw.parquet와
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_diff.parquet으로부터
# C:\Workspace\06_ML_projdect\26_1_COIN\data2\04_feature_engineering\fs_sample_28d.parquet 생성

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import math

# 1. 파일 경로 설정
BASE_DIR = r"C:\Workspace\06_ML_projdect\26_1_COIN"
IN_FILE  = os.path.join(BASE_DIR, "data2", "04_feature_engineering", "fs_sample_diff.parquet")
OUT_FILE = os.path.join(BASE_DIR, "data2", "04_feature_engineering", "fs_sample_28d.parquet")

if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)

# 2. 파케이 자체 메타데이터(Zone-Map)를 활용한 초고속 시리얼 스캔 (디스크 복사 X)
print("1/3. 원본 파케이(Parquet)에서 고유 시리얼 스캔 중...")
con = duckdb.connect()  # 임시 파일 없이 메모리에서만 가볍게 실행
serials = con.execute(f"SELECT DISTINCT serial_number FROM read_parquet('{IN_FILE.replace(chr(92), "/")}')").df()['serial_number'].tolist()

# 3. Chunk 단위 처리
CHUNK_SIZE = 1000
total_chunks = math.ceil(len(serials) / CHUNK_SIZE)
print(f"2/4. 총 {len(serials):,}개의 디스크를 {total_chunks}개의 Chunk로 안전하게 처리합니다.")

writer = None

for chunk_idx, i in enumerate(range(0, len(serials), CHUNK_SIZE)):
    chunk_serials = serials[i : i + CHUNK_SIZE]
    
    # 타입 방어적 문자열 변환
    serial_str = ", ".join([f"'{s}'" if isinstance(s, str) else str(s) for s in chunk_serials])
    

        # 4. DuckDB 조각 SQL (28d: s190/s194 가속도 제외 및 문법 수정 완료)
    sql = f"""
    WITH base AS (
        SELECT 
            serial_number, date,
            s5_diff, s187_diff, s197_diff, s198_diff,
            s183_diff, s189_diff, s191_diff, s199_diff,
            timeout_5s_diff, timeout_total_diff, seek_error_count_diff,
            s184_diff, 
            smart_190_raw, smart_194_raw, 
            s241_diff, s242_diff, 
            total_reads_diff, total_seeks_diff
        FROM read_parquet('{IN_FILE.replace(chr(92), "/")}') 
        WHERE serial_number IN (SELECT unnest([{serial_str}]))
    ),
    diffs AS (
        SELECT *,
            smart_190_raw - LAG(smart_190_raw) OVER w0 AS d_s190,
            smart_194_raw - LAG(smart_194_raw) OVER w0 AS d_s194,
            s241_diff - LAG(s241_diff) OVER w0 AS d_s241,
            s242_diff - LAG(s242_diff) OVER w0 AS d_s242,
            total_reads_diff - LAG(total_reads_diff) OVER w0 AS d_reads,
            total_seeks_diff - LAG(total_seeks_diff) OVER w0 AS d_seeks
        FROM base
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    accel_base AS (
        SELECT *,
            -- (s190, s194 가속도 제외됨)
            d_s241 - LAG(d_s241) OVER w0 AS s241_28d_accel,
            d_s242 - LAG(d_s242) OVER w0 AS s242_28d_accel,
            d_reads - LAG(d_reads) OVER w0 AS total_reads_28d_accel,
            d_seeks - LAG(d_seeks) OVER w0 AS total_seeks_28d_accel
        FROM diffs
        WINDOW w0 AS (PARTITION BY serial_number ORDER BY date)
    ),
    stats_28d AS (
        SELECT
            serial_number, date,
            SUM(s5_diff) OVER w28 AS s5_28d_sum, MAX(s5_diff) OVER w28 AS s5_28d_max,
            SUM(s187_diff) OVER w28 AS s187_28d_sum, MAX(s187_diff) OVER w28 AS s187_28d_max,
            SUM(s197_diff) OVER w28 AS s197_28d_sum, MAX(s197_diff) OVER w28 AS s197_28d_max,
            SUM(s198_diff) OVER w28 AS s198_28d_sum, MAX(s198_diff) OVER w28 AS s198_28d_max,
            SUM(s183_diff) OVER w28 AS s183_28d_sum, MAX(s183_diff) OVER w28 AS s183_28d_max,
            SUM(s189_diff) OVER w28 AS s189_28d_sum, MAX(s189_diff) OVER w28 AS s189_28d_max,
            SUM(s191_diff) OVER w28 AS s191_28d_sum, MAX(s191_diff) OVER w28 AS s191_28d_max,
            SUM(s199_diff) OVER w28 AS s199_28d_sum, MAX(s199_diff) OVER w28 AS s199_28d_max,
            SUM(timeout_5s_diff) OVER w28 AS timeout_5s_28d_sum, MAX(timeout_5s_diff) OVER w28 AS timeout_5s_28d_max,
            SUM(timeout_total_diff) OVER w28 AS timeout_total_28d_sum, MAX(timeout_total_diff) OVER w28 AS timeout_total_28d_max,
            SUM(seek_error_count_diff) OVER w28 AS seek_error_count_28d_sum, MAX(seek_error_count_diff) OVER w28 AS seek_error_count_28d_max,

            MAX(smart_190_raw) OVER w28 AS s190_28d_max, AVG(smart_190_raw) OVER w28 AS s190_28d_mean,
            COALESCE(STDDEV_SAMP(smart_190_raw) OVER w28, 0.0) AS s190_28d_std,
            SUM(ABS(d_s190)) OVER w28 AS s190_28d_asfd, SQRT(SUM(POW(d_s190, 2)) OVER w28) AS s190_28d_cid,
            (smart_190_raw - FIRST_VALUE(smart_190_raw) OVER w28_dai) / NULLIF(COUNT(smart_190_raw) OVER w28_dai - 1, 0) AS s190_28d_dai,

            MAX(smart_194_raw) OVER w28 AS s194_28d_max, AVG(smart_194_raw) OVER w28 AS s194_28d_mean,
            COALESCE(STDDEV_SAMP(smart_194_raw) OVER w28, 0.0) AS s194_28d_std,
            SUM(ABS(d_s194)) OVER w28 AS s194_28d_asfd, SQRT(SUM(POW(d_s194, 2)) OVER w28) AS s194_28d_cid,
            (smart_194_raw - FIRST_VALUE(smart_194_raw) OVER w28_dai) / NULLIF(COUNT(smart_194_raw) OVER w28_dai - 1, 0) AS s194_28d_dai,

            MAX(s241_diff) OVER w28 AS s241_28d_max, SUM(s241_diff) OVER w28 AS s241_28d_sum, AVG(s241_diff) OVER w28 AS s241_28d_mean,
            COALESCE(STDDEV_SAMP(s241_diff) OVER w28, 0.0) AS s241_28d_std, SUM(ABS(d_s241)) OVER w28 AS s241_28d_asfd,
            (s241_diff - FIRST_VALUE(s241_diff) OVER w28_dai) / NULLIF(COUNT(s241_diff) OVER w28_dai - 1, 0) AS s241_28d_dai,

            MAX(s242_diff) OVER w28 AS s242_28d_max, SUM(s242_diff) OVER w28 AS s242_28d_sum, AVG(s242_diff) OVER w28 AS s242_28d_mean,
            COALESCE(STDDEV_SAMP(s242_diff) OVER w28, 0.0) AS s242_28d_std, SUM(ABS(d_s242)) OVER w28 AS s242_28d_asfd,
            (s242_diff - FIRST_VALUE(s242_diff) OVER w28_dai) / NULLIF(COUNT(s242_diff) OVER w28_dai - 1, 0) AS s242_28d_dai,

            MAX(total_reads_diff) OVER w28 AS total_reads_28d_max, SUM(total_reads_diff) OVER w28 AS total_reads_28d_sum, AVG(total_reads_diff) OVER w28 AS total_reads_28d_mean,
            COALESCE(STDDEV_SAMP(total_reads_diff) OVER w28, 0.0) AS total_reads_28d_std, SUM(ABS(d_reads)) OVER w28 AS total_reads_28d_asfd,
            (total_reads_diff - FIRST_VALUE(total_reads_diff) OVER w28_dai) / NULLIF(COUNT(total_reads_diff) OVER w28_dai - 1, 0) AS total_reads_28d_dai,

            MAX(total_seeks_diff) OVER w28 AS total_seeks_28d_max, SUM(total_seeks_diff) OVER w28 AS total_seeks_28d_sum, AVG(total_seeks_diff) OVER w28 AS total_seeks_28d_mean,
            COALESCE(STDDEV_SAMP(total_seeks_diff) OVER w28, 0.0) AS total_seeks_28d_std, SUM(ABS(d_seeks)) OVER w28 AS total_seeks_28d_asfd,
            (total_seeks_diff - FIRST_VALUE(total_seeks_diff) OVER w28_dai) / NULLIF(COUNT(total_seeks_diff) OVER w28_dai - 1, 0) AS total_seeks_28d_dai,

            smart_190_raw AS r_190, smart_194_raw AS r_194, 
            s241_diff AS r_241, s242_diff AS r_242, 
            total_reads_diff AS r_reads, total_seeks_diff AS r_seeks,
            s241_28d_accel, s242_28d_accel, total_reads_28d_accel, total_seeks_28d_accel
        FROM accel_base
        WINDOW 
            w0 AS (PARTITION BY serial_number ORDER BY date),
            w28 AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 27 PRECEDING AND CURRENT ROW),
            w28_dai AS (PARTITION BY serial_number ORDER BY date ROWS BETWEEN 27 PRECEDING AND CURRENT ROW)
    )
    SELECT
        serial_number, date,
        s5_28d_sum, s5_28d_max, s187_28d_sum, s187_28d_max, s197_28d_sum, s197_28d_max,
        s198_28d_sum, s198_28d_max, s183_28d_sum, s183_28d_max, s189_28d_sum, s189_28d_max,
        s191_28d_sum, s191_28d_max, s199_28d_sum, s199_28d_max,
        timeout_5s_28d_sum, timeout_5s_28d_max, timeout_total_28d_sum, timeout_total_28d_max,
        seek_error_count_28d_sum, seek_error_count_28d_max,
        -- [수정] 온도 가속도(accel) 제거
        s190_28d_max, s190_28d_mean, s190_28d_std, s190_28d_asfd, s190_28d_cid, s190_28d_dai,
        CASE WHEN s190_28d_std < 0.001 THEN 0.0 ELSE (r_190 - s190_28d_mean) / (s190_28d_std + 1e-5) END AS s190_28d_zscore,
        s194_28d_max, s194_28d_mean, s194_28d_std, s194_28d_asfd, s194_28d_cid, s194_28d_dai,
        CASE WHEN s194_28d_std < 0.001 THEN 0.0 ELSE (r_194 - s194_28d_mean) / (s194_28d_std + 1e-5) END AS s194_28d_zscore,
        
        s241_28d_max, s241_28d_sum, s241_28d_mean, s241_28d_std, s241_28d_asfd, s241_28d_dai, s241_28d_accel,
        CASE WHEN s241_28d_std < 0.001 THEN 0.0 ELSE (r_241 - s241_28d_mean) / (s241_28d_std + 1e-5) END AS s241_28d_zscore,
        s242_28d_max, s242_28d_sum, s242_28d_mean, s242_28d_std, s242_28d_asfd, s242_28d_dai, s242_28d_accel,
        CASE WHEN s242_28d_std < 0.001 THEN 0.0 ELSE (r_242 - s242_28d_mean) / (s242_28d_std + 1e-5) END AS s242_28d_zscore,
        total_reads_28d_max, total_reads_28d_sum, total_reads_28d_mean, total_reads_28d_std, total_reads_28d_asfd, total_reads_28d_dai, total_reads_28d_accel,
        CASE WHEN total_reads_28d_std < 0.001 THEN 0.0 ELSE (r_reads - total_reads_28d_mean) / (total_reads_28d_std + 1e-5) END AS total_reads_28d_zscore,
        total_seeks_28d_max, total_seeks_28d_sum, total_seeks_28d_mean, total_seeks_28d_std, total_seeks_28d_asfd, total_seeks_28d_dai, total_seeks_28d_accel,
        CASE WHEN total_seeks_28d_std < 0.001 THEN 0.0 ELSE (r_seeks - total_seeks_28d_mean) / (total_seeks_28d_std + 1e-5) END AS total_seeks_28d_zscore,
        r_190, r_194, r_241, r_242, r_reads, r_seeks
    FROM stats_28d
    """


    
    df = con.query(sql).df()

    df = df.fillna(0.0)

    # EWMA 연산 오류 방지용 엄격한 시계열 정렬 적용
    df.sort_values(by=['serial_number', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # EWMA 계산 (span=28)
    ewma_targets = {
        'r_190': 's190_28d_ewma', 'r_194': 's194_28d_ewma', 'r_241': 's241_28d_ewma',
        'r_242': 's242_28d_ewma', 'r_reads': 'total_reads_28d_ewma', 'r_seeks': 'total_seeks_28d_ewma'
    }
    for col_raw, col_ewma in ewma_targets.items():
        df[col_ewma] = df.groupby('serial_number')[col_raw].transform(lambda x: x.ewm(span=28, adjust=False).mean())
    df.drop(columns=list(ewma_targets.keys()), inplace=True)

    # 강제 메모리 압축 (Float64 -> Float32) 단, 정밀도 필수 컬럼은 제외!
    exclude_keywords = ['241', '242', 'reads', 'seeks', 'cid']
    float_cols = df.select_dtypes(include=['float64']).columns
    target_cols = [c for c in float_cols if not any(k in c.lower() for k in exclude_keywords)]
    
    df[target_cols] = df[target_cols].astype('float32')

    # Parquet 디스크 쓰기
    table = pa.Table.from_pandas(df)
    if writer is None:
        writer = pq.ParquetWriter(OUT_FILE, table.schema, compression='zstd')
    writer.write_table(table)
    
    del df, table
    gc.collect()
    
    print(f"진행 상황: [{chunk_idx + 1}/{total_chunks}] 28d 조각 완료")

if writer:
    writer.close()
con.close()

print(f"3/3. 🎉 디스크 100% 이슈가 완벽히 해결된 28일 파이프라인 종료! (저장: {OUT_FILE})")

1/3. 원본 파케이(Parquet)에서 고유 시리얼 스캔 중...
2/4. 총 53,057개의 디스크를 54개의 Chunk로 안전하게 처리합니다.
진행 상황: [1/54] 28d 조각 완료
진행 상황: [2/54] 28d 조각 완료
진행 상황: [3/54] 28d 조각 완료
진행 상황: [4/54] 28d 조각 완료
진행 상황: [5/54] 28d 조각 완료
진행 상황: [6/54] 28d 조각 완료
진행 상황: [7/54] 28d 조각 완료
진행 상황: [8/54] 28d 조각 완료
진행 상황: [9/54] 28d 조각 완료
진행 상황: [10/54] 28d 조각 완료
진행 상황: [11/54] 28d 조각 완료
진행 상황: [12/54] 28d 조각 완료
진행 상황: [13/54] 28d 조각 완료
진행 상황: [14/54] 28d 조각 완료
진행 상황: [15/54] 28d 조각 완료
진행 상황: [16/54] 28d 조각 완료
진행 상황: [17/54] 28d 조각 완료
진행 상황: [18/54] 28d 조각 완료
진행 상황: [19/54] 28d 조각 완료
진행 상황: [20/54] 28d 조각 완료
진행 상황: [21/54] 28d 조각 완료
진행 상황: [22/54] 28d 조각 완료
진행 상황: [23/54] 28d 조각 완료
진행 상황: [24/54] 28d 조각 완료
진행 상황: [25/54] 28d 조각 완료
진행 상황: [26/54] 28d 조각 완료
진행 상황: [27/54] 28d 조각 완료
진행 상황: [28/54] 28d 조각 완료
진행 상황: [29/54] 28d 조각 완료
진행 상황: [30/54] 28d 조각 완료
진행 상황: [31/54] 28d 조각 완료
진행 상황: [32/54] 28d 조각 완료
진행 상황: [33/54] 28d 조각 완료
진행 상황: [34/54] 28d 조각 완료
진행 상황: [35/54] 28d 조각 완료
진행 상황: [36/54] 28d 조각 완료
진행 상황: [37/54] 28d 조각 완료
진